# Transformer residuals on the HAR + OVX anchor (Brent, 22-day volatility)

Runs `experiments/vol_residual_seq.py` from the `oilfc-residual` Kaggle dataset: a patch transformer, an
attention-free MLP and a ridge control predict what the HAR + OVX anchor misses; a weight alpha chosen on a
validation block shrinks the correction. Set **Accelerator = GPU** (Settings), then run the cells in order.
Everything is free-tier: no internet or API key is needed (an optional `arch` install enables the Model
Confidence Set). Download `residual_results.zip` from the Output panel when the last cell finishes.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import zipfile

import torch

print(
    "torch",
    torch.__version__,
    "| cuda:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
# 1. Locate the dataset (folders if Kaggle unzipped it, otherwise the zip) and copy the code to /kaggle/working
WORK = "/kaggle/working"
hits = glob.glob("/kaggle/input/**/experiments/vol_residual_seq.py", recursive=True)
if hits:
    base = os.path.dirname(os.path.dirname(hits[0]))
else:
    zp = glob.glob("/kaggle/input/**/oilfc-residual.zip", recursive=True)[0]
    base = f"{WORK}/_dataset"
    zipfile.ZipFile(zp).extractall(base)
print("dataset root:", base, os.listdir(base))
PROJ = f"{WORK}/proj"
shutil.copytree(base, PROJ, dirs_exist_ok=True)
os.chdir(PROJ)
print(os.listdir(PROJ))

In [ ]:
# 2. Optional: Model Confidence Set needs the `arch` package (skipped automatically if it is missing and there is no internet)
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "arch"], capture_output=True, text=True
)
print(
    "arch installed"
    if r.returncode == 0
    else "arch not installed (no internet?): MCS will be skipped"
)

In [ ]:
# 3. Smoke test (about a minute): 2 refits, 1 seed, 5 epochs. Fix any error here before the full run.
def run(args):
    p = subprocess.Popen(
        [
            sys.executable,
            "-u",
            "experiments/vol_residual_seq.py",
            *args,
            "--out",
            f"{WORK}/results",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        if "warn" not in line.lower():
            print(line, end="")
    assert p.wait() == 0, "run failed"


run(["--smoke", "--device", "auto"])

In [ ]:
# 4. Full run: Brent, 22-day volatility, yearly refits, 5 seeds (about 15 refits x 3 models; roughly 20-40 minutes on a T4/P100, not yet measured)
run(["--asset", "brent", "--h", "22", "--refit-months", "12", "--seeds", "5", "--device", "auto"])

In [ ]:
# 5. Optional second horizon and the WTI robustness check (comment out what you do not need)
run(["--asset", "brent", "--h", "5", "--refit-months", "12", "--seeds", "5", "--device", "auto"])
run(["--asset", "wti", "--h", "22", "--refit-months", "12", "--seeds", "5", "--device", "auto"])

In [ ]:
# 6. Package the results for download
zp = f"{WORK}/residual_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{WORK}/results/*"):
        z.write(f, os.path.basename(f))
print("ready:", zp, [os.path.basename(f) for f in glob.glob(f"{WORK}/results/*")])